In [19]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import KFold

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV

import torch
import torch.nn as nn
import torch.optim as optim


In [2]:
X_train = pd.read_csv('data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('data/X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv('data/y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv('data/sample_submission.csv',index_col='ROW_ID')

# Features

### Basic benchmark features

In [3]:
ret_cols = [c for c in X_test.columns if c.startswith("RET_")]
vol_cols = [c for c in X_test.columns if c.startswith("SIGNED_VOLUME_")]

def fillna_row_mean(df, cols):
    A = df[cols].to_numpy(dtype=float)
    m = np.isnan(A)
    if m.any():
        row_mean = np.nanmean(A, axis=1)
        # si une ligne est full-NaN (rare), remplace la "moyenne" NaN par 0
        row_mean = np.where(np.isfinite(row_mean), row_mean, 0.0)
        r, c = np.where(m)
        A[r, c] = row_mean[r]
        df.loc[:, cols] = A
    return df

X_test = fillna_row_mean(X_test, ret_cols)
X_test = fillna_row_mean(X_test, vol_cols)

In [4]:
RET_features = [f'RET_{i}' for i in range(1,20)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,20)]
TURNOVER_features = ['AVG_DAILY_TURNOVER']

In [5]:
for i in [3,5,10,15,20]:
    X_train[ f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i+1]].mean(1)
    X_train[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
    
    X_test[ f'AVERAGE_PERF_{i}'] = X_test[RET_features[:i+1]].mean(1)
    X_test[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_test.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')

In [6]:
features = RET_features + SIGNED_VOLUME_features + TURNOVER_features
features = features + [ f'AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]
features = features + [ f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]

### EMA

In [7]:
ret_cols = [f"RET_{i}" for i in range(1, 21)]
r = X_train[ret_cols].to_numpy()

rtest = X_test[ret_cols].to_numpy()

def ema(arr, L):
    alpha = 2/(L+1)
    w = (1-alpha) ** np.arange(L)  # 0..L-1
    w = w / w.sum()
    return (arr[:, :L] * w).sum(axis=1)

X_train["ema3"]  = ema(r, 3)
X_train["ema5"]  = ema(r, 5)
X_train["ema10"] = ema(r,10)


X_test["ema3"]  = ema(rtest, 3)
X_test["ema5"]  = ema(rtest, 5)
X_test["ema10"] = ema(rtest,10)

m20 = r.mean(axis=1)
s20 = r.std(axis=1, ddof=0)

m20test = rtest.mean(axis=1)
s20test = rtest.std(axis=1, ddof=0)

X_train["z20"] = m20 / (s20 + 1e-12)
X_test["z20"] = m20test / (s20test + 1e-12)

### Streak de signe et entropie 

In [ ]:
sign = np.sign(r)  
signtest = np.sign(rtest)

def last_streak_len(sig_row, positive=True):
    # part de RET_1 vers RET_20
    target = 1 if positive else -1
    cnt = 0
    for v in sig_row[:20]: 
        if v == target:
            cnt += 1
        else:
            break
    return cnt

X_train["streak_pos"] = [last_streak_len(s, True) for s in sign]
X_train["streak_neg"] = [last_streak_len(s, False) for s in sign]

X_test["streak_pos"] = [last_streak_len(s, True) for s in signtest]
X_test["streak_neg"] = [last_streak_len(s, False) for s in signtest]

p = (r > 0).mean(axis=1)
p = np.clip(p, 1e-9, 1 - 1e-9)
X_train["sign_entropy20"] = -(p*np.log(p) + (1-p)*np.log(1-p))

ptest = (rtest > 0).mean(axis=1)    
ptest = np.clip(ptest, 1e-9, 1 - 1e-9)
X_test["sign_entropy20"] = -(ptest*np.log(ptest) + (1-ptest)*np.log(1-ptest))


### Qualité de rendement (Sharpe10, Sortino10, t-stat(mean10))

In [9]:
r10 = r[:, :10]
m10 = r10.mean(axis=1)
s10 = r10.std(axis=1, ddof=0)
neg10 = np.minimum(r10, 0)

r10test = rtest[:, :10]
m10test = r10test.mean(axis=1)
s10test = r10test.std(axis=1, ddof=0)
neg10test = np.minimum(r10test, 0)

X_train["sharpe10"]   = m10 / (s10 + 1e-12)
downside10 = np.sqrt((neg10**2).mean(axis=1))
X_train["sortino10"]  = m10 / (downside10 + 1e-12)
X_train["tstat_mean10"] = m10 / (s10/np.sqrt(10) + 1e-12)

X_test["sharpe10"]   = m10test / (s10test + 1e-12)
downside10test = np.sqrt((neg10test**2).mean(axis=1))
X_test["sortino10"]  = m10test / (downside10test + 1e-12)
X_test["tstat_mean10"] = m10test / (s10test/np.sqrt(10) + 1e-12)


### Max drawdown et recovery

In [10]:
X_train["vol10"]         = s10
X_train["downside_dev10"]= downside10

X_test["vol10"]         = s10test
X_test["downside_dev10"]= downside10test

def maxdd_and_recovery(row):
    c = np.cumsum(row)                   # equity curve 20j
    peak = np.maximum.accumulate(c)
    dd = (c - peak)
    maxdd = dd.min()                     # drawdown (négatif)
    # recovery: jours depuis le dernier pic
    last_peak_idx = np.where(c == peak)[0][-1]
    recovery = 20 - 1 - last_peak_idx
    return maxdd, recovery

md_rec = np.apply_along_axis(maxdd_and_recovery, 1, r)
X_train["maxdd20"]   = md_rec[:,0]
X_train["recovery20"]= md_rec[:,1]

md_rec_test = np.apply_along_axis(maxdd_and_recovery, 1, rtest)
X_test["maxdd20"]    = md_rec_test[:,0]
X_test["recovery20"] = md_rec_test[:,1]


### Liquidity / turnover (moyenne/écart-type volumes, autocorr, corr ret–liq)

In [ ]:

ret_cols = [f"RET_{i}" for i in range(1, 21)]
vol_cols = [f"SIGNED_VOLUME_{i}" for i in range(1, 21)]
EPS = 1e-12

def add_volume_features(X):
    v = np.nan_to_num(X[vol_cols].to_numpy(), nan=0.0)
    r = np.nan_to_num(X[ret_cols].to_numpy(), nan=0.0)

    X["sv_mean20"] = v.mean(axis=1)
    X["sv_std20"]  = v.std(axis=1, ddof=0)

    v_mean = v.mean(axis=1, keepdims=True)
    vx = v - v_mean
    num = (vx[:,1:] * vx[:,:-1]).sum(axis=1)
    den = np.sqrt((vx[:,1:]**2).sum(axis=1) * (vx[:,:-1]**2).sum(axis=1))
    X["sv_autocorr1"] = num / (den + EPS)

    rx = r - r.mean(axis=1, keepdims=True)
    num = (rx * vx).sum(axis=1)
    den = np.sqrt((rx**2).sum(axis=1) * (vx**2).sum(axis=1))
    X["corr_ret_sv20"] = num / (den + EPS)

    return X

X_train = add_volume_features(X_train.copy())
X_test  = add_volume_features(X_test.copy())


In [13]:
features = [col for col in X_train.columns if col not in ['TS', 'ALLOCATION']]

# Preprocessing

### Winsorizer et SVD

In [ ]:


class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.005, upper=0.995):
        self.lower = lower; self.upper = upper
    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.lo_ = X.quantile(self.lower)
        self.hi_ = X.quantile(self.upper)
        return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return X.clip(self.lo_, self.hi_, axis=1).values

# ---- group SVD encoder (fit on train groups, transform new data) ----
class GroupSVDEncoder:
    """
    Fit: on train_df -> groupby(group_col)[feature_cols].agg('mean') -> scale -> SVD
    Transform: takes any df, computes group means in *that* df, scales with train scaler, projects with train SVD,
               then merges per-row by group id.
    """
    def __init__(self, group_col, feature_cols, n_components=8, random_state=42, agg='mean', suffix=''):
        self.group_col = group_col
        self.feature_cols = feature_cols
        self.n_components = n_components
        self.random_state = random_state
        self.agg = agg
        self.suffix = suffix or group_col.lower()

    def fit(self, df_train):
        g = getattr(df_train[self.feature_cols + [self.group_col]].groupby(self.group_col), self.agg)()
        self.scaler_ = StandardScaler().fit(g.values)
        G = self.scaler_.transform(g.values)
        self.svd_ = TruncatedSVD(n_components=self.n_components, random_state=self.random_state)
        Z = self.svd_.fit_transform(G)
        self.cols_ = [f"{self.suffix}_svd_{i+1}" for i in range(self.n_components)]
        self.train_group_emb_ = pd.DataFrame(Z, index=g.index, columns=self.cols_)
        self.ev_ = self.svd_.explained_variance_ratio_.sum()
        return self

    def transform(self, df_any):
        # build embeddings for groups present in df_any (can be unseen groups)
        g_any = getattr(df_any[self.feature_cols + [self.group_col]].groupby(self.group_col), self.agg)()
        Gt = self.scaler_.transform(g_any.values)  # scale w/ train scaler
        Zt = self.svd_.transform(Gt)
        emb = pd.DataFrame(Zt, index=g_any.index, columns=self.cols_)
        emb[self.suffix + "_svd_ratio"] = self.ev_
        # merge back per row (left join on group id)
        return df_any.merge(emb.reset_index(), on=self.group_col, how="left")


### Denoizing AutoEncoder Transformer

In [ ]:


use_dae = True

class DAETransformer(BaseEstimator, TransformerMixin):
    """
    Scikit-learn compatible DAE:
    - standardize inputs
    - add Gaussian noise during training
    - return latent code concatenated to original inputs (or just code if return_code_only=True)
    """
    def __init__(self, n_hidden=128, code_dim=32, noise_std=0.05, epochs=10, batch_size=512, lr=1e-3,
                 return_code_only=True, device='auto', random_state=42):
        self.n_hidden = n_hidden
        self.code_dim = code_dim
        self.noise_std = noise_std
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.return_code_only = return_code_only
        self.device = device
        self.random_state = random_state

    def _build(self, in_dim):
        enc = nn.Sequential(
            nn.Linear(in_dim, self.n_hidden), nn.ReLU(),
            nn.Linear(self.n_hidden, self.code_dim)
        )
        dec = nn.Sequential(
            nn.Linear(self.code_dim, self.n_hidden), nn.ReLU(),
            nn.Linear(self.n_hidden, in_dim)
        )
        return enc, dec

    def fit(self, X, y=None):
        assert use_dae, "PyTorch not available; set use_dae=True only if torch is installed."
        rs = np.random.RandomState(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        self.scaler_ = StandardScaler().fit(X)
        Xs = self.scaler_.transform(X).astype(np.float32)
        in_dim = Xs.shape[1]
        self.encoder_, self.decoder_ = self._build(in_dim)
        dev = torch.device('cuda' if (self.device=='auto' and torch.cuda.is_available()) else 'cpu')
        self.encoder_.to(dev); self.decoder_.to(dev)
        opt = optim.Adam(list(self.encoder_.parameters())+list(self.decoder_.parameters()), lr=self.lr)
        loss_fn = nn.MSELoss()

        ds = torch.utils.data.TensorDataset(torch.from_numpy(Xs))
        dl = torch.utils.data.DataLoader(ds, batch_size=self.batch_size, shuffle=True, drop_last=False)

        self.encoder_.train(); self.decoder_.train()
        for _ in range(self.epochs):
            for (xb,) in dl:
                xb = xb.to(dev)
                noise = torch.from_numpy(rs.normal(0, self.noise_std, xb.shape).astype(np.float32)).to(dev)
                xnoisy = xb + noise
                code = self.encoder_(xnoisy)
                recon = self.decoder_(code)
                loss = loss_fn(recon, xb)
                opt.zero_grad(); loss.backward(); opt.step()
        self.encoder_.eval()
        self.device_ = dev
        return self

    def transform(self, X):
        if not use_dae:
            return np.asarray(X)  # no-op fallback
        X = np.asarray(X, dtype=np.float32)
        Xs = self.scaler_.transform(X).astype(np.float32)
        with torch.no_grad():
            codes = self.encoder_(torch.from_numpy(Xs).to(self.device_)).cpu().numpy()
        if self.return_code_only:
            return codes
        return np.concatenate([X, codes], axis=1)


# Ridge training - Kfold CV

In [16]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

n_splits = 10
#kf = KFold(n_splits=n_splits, shuffle=True, random_state=43)


groups = X_train["TS"]  

train_dates = X_train['TS'].unique()
scores = []
models = []
best_rounds = []
best_scores = []

y_train_bin = y_train.copy()
y_train_bin["target"] = (y_train_bin["target"] > 0).astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
(tr_idx_all, hold_idx) = next(gss.split(X_train, y_train, groups))
X_cv, y_cv = X_train.iloc[tr_idx_all].reset_index(drop=True), y_train.iloc[tr_idx_all].reset_index(drop=True)
X_hold, y_hold = X_train.iloc[hold_idx].reset_index(drop=True), y_train.iloc[hold_idx].reset_index(drop=True)

# OOF et HOLDOUT preds containers
oof_pred = np.full(len(X_train), np.nan)
oof_cls  = np.zeros(len(X_train), dtype=int)
fold_acc = []
fold_models = []
fold_selectors = []
fold_thresholds = []
hold_preds = []  


hold_votes = np.zeros((n_splits, len(X_hold)), dtype=int)
test_votes = np.zeros((n_splits, len(X_test)), dtype=int)


gkf = GroupKFold(n_splits=n_splits)


In [17]:
X_hold.columns

Index(['TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17', 'RET_16',
       'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10', 'RET_9',
       'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2', 'RET_1',
       'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'AVG_DAILY_TURNOVER',
       'AVERAGE_PERF_3', 'ALLOCATIONS_AVERAGE_PERF_3', 'AVERAGE_PERF_5',
       'ALLOCATIONS_AVERAGE_PERF_5', 'AVERAGE_PERF_10',
       'ALLOCATIONS_AVERAGE_PERF_10', 'AVERAGE_PERF_15',
       'ALLOCATIONS_AVERAGE_PERF_15', 'AVERAGE_PERF_20',
       'ALLOCATIONS_AVERAGE_PERF_20', 'ema3', 'ema5', 'ema10', 

In [18]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold, GroupKFold
from sklearn.preprocessing import StandardScaler

class RobustFeatureSelector(BaseEstimator, TransformerMixin):
    """
    Ultra-simple robust selector:
      1) Split data into K folds (GroupKFold if groups provided)
      2) For each fold, compute per-feature Spearman rho(X_j, y) on the train part
      3) Aggregate per-feature |rho| by a robust quantile (stability_q, e.g., median)
      4) Sort features by score desc, then greedily drop features highly correlated
         (|Pearson| >= col_thresh) with already-selected ones
      5) Keep top max_keep features

    Parameters
    ----------
    groups : array-like or None
        If provided, GroupKFold(groups) is used.
    k_folds : int
        Number of folds for robust scoring.
    col_thresh : float
        Max allowed absolute correlation between kept features (redundancy filter).
    stability_q : float in (0,1]
        Quantile to aggregate |rho| across folds (e.g., 0.5 = median).
    max_keep : int
        Max number of features to keep.
    standardize : bool
        If True, standardize X before correlation checks (helps redundancy step).
    random_state : int
        Seed for shuffled KFold.

    Notes
    -----
    - Accepts numeric columns only; non-numeric are dropped.
    - Missing values are imputed by training-fold medians.
    - Returns a pandas DataFrame with selected columns (keeps names).
    """
    def __init__(self,
                 groups=None,
                 k_folds=5,
                 col_thresh=0.95,
                 stability_q=0.5,
                 max_keep=200,
                 standardize=True,
                 random_state=42):
        self.groups = groups
        self.k_folds = k_folds
        self.col_thresh = col_thresh
        self.stability_q = stability_q
        self.max_keep = max_keep
        self.standardize = standardize
        self.random_state = random_state

    def _ensure_df(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X).copy()

    def fit(self, X, y):
        X = self._ensure_df(X)
        y = np.asarray(y).ravel()

        # keep only numeric columns
        num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
        X = X[num_cols]
        self.feature_names_in_ = num_cols

        n = len(X)
        groups = None
        if self.groups is not None:
            groups = np.asarray(self.groups)
            assert len(groups) == n, "groups length must match X."

        # splitter
        if groups is not None:
            splitter = GroupKFold(n_splits=self.k_folds)
            split_iter = splitter.split(X, y, groups=groups)
        else:
            splitter = KFold(n_splits=self.k_folds, shuffle=True, random_state=self.random_state)
            split_iter = splitter.split(X, y)

        # compute per-fold medians for imputation + per-fold Spearman |rho|
        rhos = []  # list of arrays shape (n_features,)
        for tr_idx, _ in split_iter:
            Xtr = X.iloc[tr_idx]
            ytr = y[tr_idx]

            # median impute (fit on train fold)
            med = Xtr.median(axis=0)
            Xtr_imp = Xtr.fillna(med)

            # Spearman |rho|, per feature (simple & readable)
            vals = []
            rank_y = pd.Series(ytr).rank(method="average").to_numpy()
            for c in Xtr_imp.columns:
                x = Xtr_imp[c].to_numpy()
                # rank x (Spearman)
                rank_x = pd.Series(x).rank(method="average").to_numpy()
                # pearson on ranks
                vx = rank_x - rank_x.mean()
                vy = rank_y - rank_y.mean()
                denom = np.sqrt((vx**2).sum() * (vy**2).sum())
                if denom == 0:
                    rho = 0.0
                else:
                    rho = (vx @ vy) / denom
                vals.append(abs(rho))
            rhos.append(np.asarray(vals, dtype=float))

        R = np.vstack(rhos)  # (k_folds, n_features)
        # robust aggregate score per feature
        q = np.clip(self.stability_q, 0.0, 1.0)
        agg = np.quantile(R, q=q, axis=0)  # shape (n_features,)

        # sort features by score desc
        order = np.argsort(-agg)
        sorted_feats = [self.feature_names_in_[i] for i in order]
        sorted_scores = agg[order]

        # prepare dataset for redundancy filtering (global medians + optional standardize)
        med_global = X.median(axis=0)
        X_imp = X.fillna(med_global)
        if self.standardize:
            scaler = StandardScaler(with_mean=True, with_std=True)
            X_std = pd.DataFrame(scaler.fit_transform(X_imp), columns=X_imp.columns, index=X_imp.index)
        else:
            X_std = X_imp

        # greedy redundancy filter
        kept = []
        for f in sorted_feats:
            if len(kept) >= self.max_keep:
                break
            keep = True
            xf = X_std[f]
            for g in kept:
                corr = float(np.corrcoef(xf, X_std[g])[0, 1])
                if np.isnan(corr):
                    corr = 0.0
                if abs(corr) >= self.col_thresh:
                    keep = False
                    break
            if keep:
                kept.append(f)

        self.selected_features_ = kept
        self.scores_ = {feat: float(score) for feat, score in zip(sorted_feats, sorted_scores)}
        self.medians_ = med_global
        self.scaler_ = None
        if self.standardize:
            self.scaler_ = StandardScaler(with_mean=True, with_std=True).fit(X_imp[self.selected_features_])
        return self

    def transform(self, X):
        X = self._ensure_df(X)
        # only keep columns seen during fit; silently drop missing
        cols = [c for c in self.selected_features_ if c in X.columns]
        X = X[cols].copy()

        # impute with training medians (for those columns)
        for c in cols:
            if c not in self.medians_:
                continue
            
            X[c] = X[c].fillna(self.medians_[c])

        # (optional) standardize to the training scaler (kept features only)
        if self.standardize and self.scaler_ is not None:
            X.loc[:, cols] = self.scaler_.transform(X[cols])

        return X

    # convenience helper
    def get_support(self):
        """Boolean mask aligned to feature_names_in_ (True if kept)."""
        keep = set(self.selected_features_)
        return np.array([c in keep for c in self.feature_names_in_], dtype=bool)

    def get_feature_names_out(self):
        return np.array(self.selected_features_, dtype=object)


In [ ]:
# Assumptions:
# - X_train, y_train, X_test, X_hold already defined
# - y_train has column "target" (as in your code)
# - gkf = GroupKFold(...) already defined
# - RobustFeatureSelector class is available
# - selected_features, oof_pred, oof_cls, test_votes, fold_models, fold_selectors, fold_thresholds, fold_acc arrays/lists exist

i = 1
selected_features = []

# ridge alpha grid (yours)
ALPHAS = np.logspace(0, 15, 21)

# choose columns
ALL_COLS = [c for c in X_train.columns if c not in ["TS","ALLOCATION","target"]]
# split by dtype (roughly)
num_cols = [c for c in ALL_COLS if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in ALL_COLS if c not in num_cols]

# base numeric preprocessor: impute + winsorize + robust scale
num_pre = Pipeline([
    ("imp", SimpleImputer(strategy="median", add_indicator=True)),
    ("win", Winsorizer(0.005, 0.995)),
    ("scale", RobustScaler(with_centering=True, with_scaling=True))
])

# if you want to one-hot cats before DAE/selector (optional; often helpful)
from sklearn.preprocessing import OneHotEncoder
cat_pre = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=10, sparse_output=False))
])

def preprocess_numeric(df):
    """Fit-independent: returns numeric matrix and fitted transformer if requested."""
    return num_pre.fit_transform(df[num_cols])

def preprocess_cats_fit(df):
    return cat_pre.fit(df[cat_cols])

def preprocess_cats_transform(fitted, df):
    if not cat_cols: return np.zeros((len(df),0))
    return fitted.transform(df[cat_cols])

# DAE config (latent appended *after* basic preprocessing; unsupervised)
dae = DAETransformer(n_hidden=128, code_dim=32, noise_std=0.05, epochs=12, batch_size=1024, lr=1e-3,
                     return_code_only=True, random_state=42) if use_dae else None

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train["TS"])):
    print(f"[META] new meta fold {i}")
    x_tr, y_tr = X_train.iloc[tr_idx].copy(), y_train.iloc[tr_idx].copy()
    x_val, y_val = X_train.iloc[val_idx].copy(), y_train.iloc[val_idx].copy()
    x_test = X_test.copy()
    x_hold = X_hold.copy()

    y_val_bin = (y_val["target"] > 0).astype(int)

    # ---------- leak-safe preprocessing: fit on train fold ----------
    # cats
    ALL_COLS = [c for c in X_train.columns if c not in ["TS", "ALLOCATION", "target"]]

    # keep only numeric, non-all-NaN columns
    num_cols = [c for c in ALL_COLS
                if pd.api.types.is_numeric_dtype(X_train[c]) and X_train[c].notna().any()]

    cat_cols = []  # explicitly none

    # --- numeric preprocessor only ---
    num_pre = Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("win", Winsorizer(0.005, 0.995)),
        ("scale", RobustScaler(with_centering=True, with_scaling=True)),
    ])

    # --- inside each fold (replace your cat section entirely) ---

    # fit numeric preprocessor on train fold only
    num_pre_fit = clone(num_pre).fit(x_tr[num_cols])
    Xtr_num  = num_pre_fit.transform(x_tr[num_cols])
    Xval_num = num_pre_fit.transform(x_val[num_cols])
    Xte_num  = num_pre_fit.transform(x_test[num_cols])
    Xhold_num= num_pre_fit.transform(x_hold[num_cols])

    # allocation / TS encoders still use numeric columns
    r_alloc = 8
    alloc_encoder = GroupSVDEncoder(group_col="ALLOCATION",
                                    feature_cols=num_cols, n_components=r_alloc, suffix="alloc").fit(x_tr)
    x_tr  = alloc_encoder.transform(x_tr)
    x_val = alloc_encoder.transform(x_val)
    x_test= alloc_encoder.transform(x_test)
    x_hold= alloc_encoder.transform(x_hold)
    alloc_cols = [c for c in x_tr.columns if c.startswith("alloc_svd_")] + ["alloc_svd_ratio"]

    r_ts = 8
    ts_encoder = GroupSVDEncoder(group_col="TS",
                                feature_cols=num_cols, n_components=r_ts, suffix="ts").fit(x_tr)
    x_tr  = ts_encoder.transform(x_tr)
    x_val = ts_encoder.transform(x_val)
    x_test= ts_encoder.transform(x_test)
    x_hold= ts_encoder.transform(x_hold)
    ts_cols = [c for c in x_tr.columns if c.startswith("ts_svd_")] + ["ts_svd_ratio"]

    # --- build matrices (no X*_cat now) ---
    Xtr = np.column_stack([Xtr_num,  x_tr[alloc_cols].to_numpy(),  x_tr[ts_cols].to_numpy()])
    Xval= np.column_stack([Xval_num, x_val[alloc_cols].to_numpy(), x_val[ts_cols].to_numpy()])
    Xte = np.column_stack([Xte_num,  x_test[alloc_cols].to_numpy(), x_test[ts_cols].to_numpy()])
    XholdM = np.column_stack([Xhold_num, x_hold[alloc_cols].to_numpy(), x_hold[ts_cols].to_numpy()])

    # --- DAE (unchanged) ---
    if use_dae:
        dae_fit = clone(dae).fit(Xtr)
        Ztr, Zval, Zte, Zhold = (dae_fit.transform(Xtr),
                                dae_fit.transform(Xval),
                                dae_fit.transform(Xte),
                                dae_fit.transform(XholdM))
        Xtr   = np.column_stack([Xtr, Ztr])
        Xval  = np.column_stack([Xval, Zval])
        Xte   = np.column_stack([Xte,  Zte])
        XholdM= np.column_stack([XholdM, Zhold])

    # --- selector + ridge as before ---
    feat_names = [f"f_{k}" for k in range(Xtr.shape[1])]
    Xtr_df  = pd.DataFrame(Xtr,  columns=feat_names)
    Xval_df = pd.DataFrame(Xval, columns=feat_names)
    Xte_df  = pd.DataFrame(Xte,  columns=feat_names)

    selector = RobustFeatureSelector(
        groups=X_train.loc[tr_idx, "TS"],
        k_folds=3, col_thresh=0.95, stability_q=0.4, max_keep=200, standardize=True,
    )
    selector.fast_spearman = True
    selector.mi_subsample_rows = 20000

    with np.errstate(all='ignore'):
        Xtr_sel  = selector.fit_transform(Xtr_df, np.asarray(y_tr).ravel())
        Xval_sel = selector.transform(Xval_df)
        Xte_sel  = selector.transform(Xte_df)
        ridge = RidgeCV(alphas=ALPHAS, cv=5, scoring="neg_mean_squared_error", fit_intercept=True).fit(Xtr_sel, y_tr)

    # ---- predictions / threshold search ----
    p_val = ridge.predict(Xval_sel)

    cand = np.unique(np.quantile(p_val, np.linspace(0.05, 0.95, 31)))
    cand = np.append(cand, 0.0)
    accs = [(thr, accuracy_score(y_val_bin, (p_val >= thr).astype(int))) for thr in cand]
    t_star, acc_star = max(accs, key=lambda x: x[1])

    oof_pred[val_idx] = p_val
    oof_cls[val_idx]  = (p_val >= t_star).astype(int)
    fold_acc.append(acc_star)

    # ---- test votes (and hold, if used) ----
    p_tst = ridge.predict(Xte_sel)
    test_votes[i-1, :] = (p_tst >= t_star).astype(int)

    # stock
    fold_models.append(ridge)
    fold_selectors.append(selector)
    fold_thresholds.append(t_star)

    print(f"[META] Fold {i:02d} | kept={Xtr_sel.shape[1]} | alpha*={ridge.alpha_:.4g} | thr*={t_star:.4g} | Acc(val)={acc_star*100:.2f}%")
    i += 1

oof_acc = accuracy_score((y_train["target"]>0).astype(int), oof_cls)
print(f"OOF Accuracy = {oof_acc*100:.2f}% | mean(fold Acc) = {np.mean(fold_acc)*100:.2f}% ± {np.std(fold_acc)*100:.2f}%")

# majority vote on test (already in test_votes)
# y_hold_hat_cls = (test_votes.mean(axis=0) >= 0.5).astype(int)


[META] new meta fold 1


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/alexandrem

In [ ]:
# ---- accuracy finale sur HOLD (X_hold, y_hold) ----
from sklearn.metrics import accuracy_score
y_hold_bin = (np.asarray(y_hold).ravel() > 0).astype(int)

# vote majoritaire des folds

hold_acc = accuracy_score(y_hold_bin, y_hold_hat_cls)
print(f"\nFinal HOLD Accuracy (majority vote) = {hold_acc*100:.2f}% | ")
     # f"mean(fold-on-hold) = {np.mean(hold_fold_acc)*100:.2f}% ± {np.std(hold_fold_acc)*100:.2f}%")


NameError: name 'y_hold_hat_cls' is not defined

In [15]:
# Assumptions:
# - X_train, y_train, X_test, X_hold already defined
# - y_train has column "target" (as in your code)
# - gkf = GroupKFold(...) already defined
# - RobustFeatureSelector class is available
# - selected_features, oof_pred, oof_cls, test_votes, fold_models, fold_selectors, fold_thresholds, fold_acc arrays/lists exist

i = 1
selected_features = []

# ridge alpha grid (yours)
ALPHAS = np.logspace(0, 15, 21)

# choose columns
ALL_COLS = [c for c in X_train.columns if c not in ["TS","ALLOCATION","target"]]
# split by dtype (roughly)
num_cols = [c for c in ALL_COLS if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in ALL_COLS if c not in num_cols]

# base numeric preprocessor: impute + winsorize + robust scale
num_pre = Pipeline([
    ("imp", SimpleImputer(strategy="median", add_indicator=True)),
    ("win", Winsorizer(0.005, 0.995)),
    ("scale", RobustScaler(with_centering=True, with_scaling=True))
])

# if you want to one-hot cats before DAE/selector (optional; often helpful)


# DAE config (latent appended *after* basic preprocessing; unsupervised)
dae = DAETransformer(n_hidden=128, code_dim=32, noise_std=0.05, epochs=12, batch_size=1024, lr=1e-3,
                     return_code_only=True, random_state=42) if use_dae else None

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train["TS"])):
    print(f"[META] new meta fold {i}")
    x_tr, y_tr = X_train.iloc[tr_idx].copy(), y_train.iloc[tr_idx].copy()
    x_val, y_val = X_train.iloc[val_idx].copy(), y_train.iloc[val_idx].copy()
    x_test = X_test.copy()
    x_hold = X_hold.copy()

    y_val_bin = (y_val["target"] > 0).astype(int)

    # ---------- leak-safe preprocessing: fit on train fold ----------
    # cats
    ALL_COLS = [c for c in X_train.columns if c not in ["TS", "ALLOCATION", "target"]]

    # keep only numeric, non-all-NaN columns
    num_cols = [c for c in ALL_COLS
                if pd.api.types.is_numeric_dtype(X_train[c]) and X_train[c].notna().any()]

    cat_cols = []  # explicitly none

    # --- numeric preprocessor only ---
    num_pre = Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("win", Winsorizer(0.005, 0.995)),
        ("scale", RobustScaler(with_centering=True, with_scaling=True)),
    ])

    # --- inside each fold (replace your cat section entirely) ---

    # fit numeric preprocessor on train fold only
    num_pre_fit = clone(num_pre).fit(x_tr[num_cols])
    Xtr_num  = num_pre_fit.transform(x_tr[num_cols])
    Xval_num = num_pre_fit.transform(x_val[num_cols])
    Xte_num  = num_pre_fit.transform(x_test[num_cols])
    Xhold_num= num_pre_fit.transform(x_hold[num_cols])

    # allocation / TS encoders still use numeric columns
    r_alloc = 8
    alloc_encoder = GroupSVDEncoder(group_col="ALLOCATION",
                                    feature_cols=num_cols, n_components=r_alloc, suffix="alloc").fit(x_tr)
    x_tr  = alloc_encoder.transform(x_tr)
    x_val = alloc_encoder.transform(x_val)
    x_test= alloc_encoder.transform(x_test)
    x_hold= alloc_encoder.transform(x_hold)
    alloc_cols = [c for c in x_tr.columns if c.startswith("alloc_svd_")] + ["alloc_svd_ratio"]

    r_ts = 8
    ts_encoder = GroupSVDEncoder(group_col="TS",
                                feature_cols=num_cols, n_components=r_ts, suffix="ts").fit(x_tr)
    x_tr  = ts_encoder.transform(x_tr)
    x_val = ts_encoder.transform(x_val)
    x_test= ts_encoder.transform(x_test)
    x_hold= ts_encoder.transform(x_hold)
    ts_cols = [c for c in x_tr.columns if c.startswith("ts_svd_")] + ["ts_svd_ratio"]

    # --- build matrices (no X*_cat now) ---
    Xtr = np.column_stack([Xtr_num,  x_tr[alloc_cols].to_numpy(),  x_tr[ts_cols].to_numpy()])
    Xval= np.column_stack([Xval_num, x_val[alloc_cols].to_numpy(), x_val[ts_cols].to_numpy()])
    Xte = np.column_stack([Xte_num,  x_test[alloc_cols].to_numpy(), x_test[ts_cols].to_numpy()])
    XholdM = np.column_stack([Xhold_num, x_hold[alloc_cols].to_numpy(), x_hold[ts_cols].to_numpy()])

    # --- DAE (unchanged) ---
    if use_dae:
        dae_fit = clone(dae).fit(Xtr)
        Ztr, Zval, Zte, Zhold = (dae_fit.transform(Xtr),
                                dae_fit.transform(Xval),
                                dae_fit.transform(Xte),
                                dae_fit.transform(XholdM))
        Xtr   = np.column_stack([Xtr, Ztr])
        Xval  = np.column_stack([Xval, Zval])
        Xte   = np.column_stack([Xte,  Zte])
        XholdM= np.column_stack([XholdM, Zhold])

    # --- selector + ridge as before ---
    feat_names = [f"f_{k}" for k in range(Xtr.shape[1])]
    Xtr_df  = pd.DataFrame(Xtr,  columns=feat_names)
    Xval_df = pd.DataFrame(Xval, columns=feat_names)
    Xte_df  = pd.DataFrame(Xte,  columns=feat_names)

    selector = RobustFeatureSelector(
        groups=X_train.loc[tr_idx, "TS"],
        k_folds=3, col_thresh=0.95, stability_q=0.4, max_keep=200, standardize=True,
    )
    selector.fast_spearman = True
    selector.mi_subsample_rows = 20000

    with np.errstate(all='ignore'):
        Xtr_sel  = selector.fit_transform(Xtr_df, np.asarray(y_tr).ravel())
        Xval_sel = selector.transform(Xval_df)
        Xte_sel  = selector.transform(Xte_df)
        ridge = RidgeCV(alphas=ALPHAS, cv=5, scoring="neg_mean_squared_error", fit_intercept=True).fit(Xtr_sel, y_tr)

    # ---- predictions / threshold search ----
    p_val = ridge.predict(Xval_sel)

    cand = np.unique(np.quantile(p_val, np.linspace(0.05, 0.95, 31)))
    cand = np.append(cand, 0.0)
    accs = [(thr, accuracy_score(y_val_bin, (p_val >= thr).astype(int))) for thr in cand]
    t_star, acc_star = max(accs, key=lambda x: x[1])

    oof_pred[val_idx] = p_val
    oof_cls[val_idx]  = (p_val >= t_star).astype(int)
    fold_acc.append(acc_star)

    # ---- test votes (and hold, if used) ----
    p_tst = ridge.predict(Xte_sel)
    test_votes[i-1, :] = (p_tst >= t_star).astype(int)

    # stock
    fold_models.append(ridge)
    fold_selectors.append(selector)
    fold_thresholds.append(t_star)

    print(f"[META] Fold {i:02d} | kept={Xtr_sel.shape[1]} | alpha*={ridge.alpha_:.4g} | thr*={t_star:.4g} | Acc(val)={acc_star*100:.2f}%")
    i += 1

oof_acc = accuracy_score((y_train["target"]>0).astype(int), oof_cls)
print(f"OOF Accuracy = {oof_acc*100:.2f}% | mean(fold Acc) = {np.mean(fold_acc)*100:.2f}% ± {np.std(fold_acc)*100:.2f}%")

# majority vote on test (already in test_votes)
# y_hold_hat_cls = (test_votes.mean(axis=0) >= 0.5).astype(int)


[META] new meta fold 1


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/alexandrem

: 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import RidgeCV
from sklearn.base import clone

# ----------------------------
# column setup (no categoricals)
# ----------------------------
ALL_COLS = [c for c in X_train.columns if c not in ["TS", "ALLOCATION", "target"]]
num_cols = [c for c in ALL_COLS
            if pd.api.types.is_numeric_dtype(X_train[c]) and X_train[c].notna().any()]

# ----------------------------
# hyperparams
# ----------------------------
ALPHAS = np.logspace(0, 15, 21)     # your grid
R_ALLOC = 8                         # alloc embedding size
R_TS = 8                            # ts embedding size
USE_DAE = True                      # set False to skip

# ----------------------------
# preprocess numeric (fit on full train)
# ----------------------------
num_pre = Pipeline([
    ("imp", SimpleImputer(strategy="median", add_indicator=True)),
    ("win", Winsorizer(0.005, 0.995)),
    ("scale", RobustScaler(with_centering=True, with_scaling=True)),
])

num_pre_fit = clone(num_pre).fit(X_train[num_cols])
Xtr_num  = num_pre_fit.transform(X_train[num_cols])
Xte_num  = num_pre_fit.transform(X_test[num_cols])

# ----------------------------
# group embeddings (fit on full train)
# ----------------------------
alloc_encoder = GroupSVDEncoder(group_col="ALLOCATION",
                                feature_cols=num_cols,
                                n_components=R_ALLOC,
                                suffix="alloc").fit(X_train)

Xtr_fe = alloc_encoder.transform(X_train)
Xte_fe = alloc_encoder.transform(X_test)
alloc_cols = [c for c in Xtr_fe.columns if c.startswith("alloc_svd_")] + ["alloc_svd_ratio"]

ts_encoder = GroupSVDEncoder(group_col="TS",
                             feature_cols=num_cols,
                             n_components=R_TS,
                             suffix="ts").fit(X_train)

Xtr_fe = ts_encoder.transform(Xtr_fe)
Xte_fe = ts_encoder.transform(Xte_fe)
ts_cols = [c for c in Xtr_fe.columns if c.startswith("ts_svd_")] + ["ts_svd_ratio"]

# ----------------------------
# build design matrices
# ----------------------------
Xtr = np.column_stack([Xtr_num,
                       Xtr_fe[alloc_cols].to_numpy(),
                       Xtr_fe[ts_cols].to_numpy()])
Xte = np.column_stack([Xte_num,
                       Xte_fe[alloc_cols].to_numpy(),
                       Xte_fe[ts_cols].to_numpy()])

# ----------------------------
# optional DAE (unsupervised, fit on full train)
# ----------------------------
if USE_DAE:
    dae = DAETransformer(n_hidden=128, code_dim=32, noise_std=0.05,
                         epochs=12, batch_size=1024, lr=1e-3,
                         return_code_only=True, random_state=42)
    dae_fit = clone(dae).fit(Xtr)
    Ztr = dae_fit.transform(Xtr)
    Zte = dae_fit.transform(Xte)
    Xtr = np.column_stack([Xtr, Ztr])
    Xte = np.column_stack([Xte,  Zte])

# ----------------------------
# robust feature selection (fit on full train)
# ----------------------------
feat_names = [f"f_{k}" for k in range(Xtr.shape[1])]
Xtr_df = pd.DataFrame(Xtr, columns=feat_names)
Xte_df = pd.DataFrame(Xte, columns=feat_names)

selector = RobustFeatureSelector(
    groups=None,          # full-data fit; no grouping since no CV step
    k_folds=3,            # still used internally for robustness of ranks
    col_thresh=0.95,
    stability_q=0.4,
    max_keep=200,
    standardize=True,
    random_state=42
)
Xtr_sel = selector.fit_transform(Xtr_df, np.asarray(y_train["target"]).ravel())
Xte_sel = selector.transform(Xte_df)

# ----------------------------
# final RidgeCV on full train
# ----------------------------
ridge = RidgeCV(alphas=ALPHAS, cv=5, scoring="neg_mean_squared_error", fit_intercept=True)
ridge.fit(Xtr_sel, y_train["target"].to_numpy().ravel())

# ----------------------------
# predictions on X_test
# ----------------------------
y_test_pred = ridge.predict(Xte_sel)

print(f"[INFO] Train used features: {Xtr_sel.shape[1]} | alpha* = {ridge.alpha_:.4g}")
print(f"[INFO] X_test shape in: {X_test.shape}, design: {Xte_sel.shape}")
# If you also want a class prediction (threshold at 0 like your task):
y_test_cls = (y_test_pred >= 0.0).astype(int)  # or choose your own threshold

# Optionally keep artifacts
artifacts = {
    "num_pre": num_pre_fit,
    "alloc_encoder": alloc_encoder,
    "ts_encoder": ts_encoder,
    "dae": dae_fit if USE_DAE else None,
    "selector": selector,
    "model": ridge,
    "selected_feature_names": list(Xtr_sel.columns),
}


In [ ]:
y_test_cls

array([0, 1, 0, ..., 1, 1, 0])

In [18]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD

class GroupSVDEncoderSafe:
    """
    Encodage SVD sur des agrégats de groupes, avec garde-fous numériques.
    - group_col: clé de groupe (ex: 'ALLOCATION' ou 'TS')
    - feature_cols: colonnes numériques à agréger (ex: num_cols)
    - n_components: tentative; sera réduite si nécessaire
    - agg: 'mean' (par défaut)
    - clip: borne absolue max avant SVD pour éviter overflow
    """
    def __init__(self, group_col, feature_cols, n_components=8,
                 agg='mean', suffix='', clip=1e6, random_state=42, n_iter=7):
        self.group_col = group_col
        self.feature_cols = list(feature_cols)
        self.n_components = int(n_components)
        self.agg = agg
        self.suffix = suffix or group_col.lower()
        self.clip = float(clip)
        self.random_state = random_state
        self.n_iter = n_iter

    def _clean_matrix(self, A: pd.DataFrame) -> pd.DataFrame:
        A = A.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        # drop colonnes constantes (std ~ 0)
        std = A.std(axis=0)
        keep = std > 0.0
        A = A.loc[:, keep]
        self.cols_kept_ = list(A.columns)
        if A.shape[1] == 0:
            # cas extrême: toutes constantes → laisser une colonne 0 pour SVD
            A = pd.DataFrame(np.zeros((A.shape[0], 1), dtype=float), index=A.index, columns=["_const0"])
            self.cols_kept_ = ["_const0"]
        # standardisation safe (std=0 → 1)
        mean = A.mean(axis=0).to_numpy()
        std = A.std(axis=0).to_numpy()
        std_safe = np.where(std == 0.0, 1.0, std)
        A = (A - mean) / std_safe
        # clip
        A = A.clip(-self.clip, self.clip)
        return A

    def fit(self, df_train: pd.DataFrame):
        # agrégats groupe x features
        gobj = getattr(df_train[self.feature_cols + [self.group_col]].groupby(self.group_col), self.agg)
        G = gobj()
        # nettoyage + standardisation
        Gc = self._clean_matrix(G[self.cols_kept_] if hasattr(self, "cols_kept_") else G[self.feature_cols])
        # n_components safe
        max_rank = max(1, min(Gc.shape[0], Gc.shape[1]))
        self.n_comp_ = int(min(self.n_components, max_rank))
        # SVD
        self.svd_ = TruncatedSVD(n_components=self.n_comp_, algorithm="randomized",
                                 n_iter=self.n_iter, random_state=self.random_state)
        Z = self.svd_.fit_transform(Gc.values.astype(np.float64))
        self.emb_cols_ = [f"{self.suffix}_svd_{i+1}" for i in range(self.n_comp_)]
        self.ev_ = float(self.svd_.explained_variance_ratio_.sum())
        self.train_index_ = G.index  # groupes vus à l'entraînement
        self._train_emb_ = pd.DataFrame(Z, index=Gc.index, columns=self.emb_cols_)
        return self

    def transform(self, df_any: pd.DataFrame) -> pd.DataFrame:
        # recompute agrégats pour les groupes présents dans df_any
        gobj = getattr(df_any[self.feature_cols + [self.group_col]].groupby(self.group_col), self.agg)
        Gt = gobj()
        # réindexer sur les mêmes colonnes que train (celles gardées)
        if hasattr(self, "cols_kept_"):
            Gt = Gt.reindex(columns=self.cols_kept_, fill_value=0.0)
        # nettoyage / standardisation avec les stats de Gt (non supervisé)
        Gt = Gt.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        # standardisation safe (sur Gt lui-même — pas de fuite, c’est non supervisé)
        mean = Gt.mean(axis=0).to_numpy()
        std = Gt.std(axis=0).to_numpy()
        std_safe = np.where(std == 0.0, 1.0, std)
        Gt = (Gt - mean) / std_safe
        Gt = Gt.clip(-self.clip, self.clip)

        # si trop peu de dims, réduire n_components à la volée
        max_rank_t = max(1, min(Gt.shape[0], Gt.shape[1]))
        n_comp_t = int(min(self.n_comp_, max_rank_t))
        if n_comp_t != self.n_comp_:
            # on crée une SVD à la volée pour projeter proprement
            svd_t = TruncatedSVD(n_components=n_comp_t, algorithm="randomized",
                                 n_iter=self.n_iter, random_state=self.random_state)
            Zt = svd_t.fit_transform(Gt.values.astype(np.float64))
            emb_cols = [f"{self.suffix}_svd_{i+1}" for i in range(n_comp_t)]
        else:
            Zt = self.svd_.transform(Gt.values.astype(np.float64))
            emb_cols = self.emb_cols_

        emb = pd.DataFrame(Zt, index=Gt.index, columns=emb_cols)
        emb[self.suffix + "_svd_ratio"] = self.ev_

        # merge par ligne
        out = df_any.merge(emb.reset_index(), on=self.group_col, how="left")
        # fillna pour les groupes complètement inconnus
        fill_cols = emb_cols + [self.suffix + "_svd_ratio"]
        out[fill_cols] = out[fill_cols].fillna(0.0)
        return out


In [31]:
# --- construire la submission ---
#assert "ROW_ID" in X_test.columns, "La colonne ROW_ID doit exister dans X_test."
assert len(y_test_pred) == len(X_test), "Taille predictions ≠ taille X_test."

sub = pd.DataFrame({
    "ROW_ID": X_test.index,
    "prediction": (np.asarray(y_test_pred, dtype=float) > 0).astype(int)
})

# (optionnel) sécurité anti-NaN / inf
sub["prediction"] = np.nan_to_num(sub["prediction"], nan=0.0, posinf=0.0, neginf=0.0)

# --- sauvegarde ---
sub.to_csv("data/superlinear.csv", index=False)
print(sub.head())
print(f"[OK] submission.csv écrit avec {len(sub):,} lignes.")


   ROW_ID  prediction
0  180245           0
1  180246           1
2  180247           0
3  180248           0
4  180249           0
[OK] submission.csv écrit avec 7,735 lignes.


In [20]:
# ==== OPTUNA: 30 essais, GroupKFold CV, pruning ====
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
import optuna
from math import log10

# ---------- colonnes (num uniquement) ----------
ALL_COLS = [c for c in X_train.columns if c not in ["TS", "ALLOCATION", "target", "ROW_ID"]]
num_cols = [c for c in ALL_COLS
            if pd.api.types.is_numeric_dtype(X_train[c]) and X_train[c].notna().any()]
assert len(num_cols) > 0, "Aucune colonne numérique détectée dans X_train."

y_full = (y_train["target"].to_numpy().ravel()
          if isinstance(y_train, pd.DataFrame) else np.asarray(y_train).ravel())

# ---------- helpers: construire design matricies ----------
def make_num_pre(lower_q, upper_q, scaler_type):
    scaler = RobustScaler(with_centering=True, with_scaling=True) if scaler_type=="robust" \
             else StandardScaler(with_mean=True, with_std=True)
    return Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("win", Winsorizer(lower_q, upper_q)),
        ("scale", scaler),
    ])

def build_design(x_tr_df, x_te_df, params):
    """
    Retourne (Xtr, Xte) numpy arrays:
      - num preprocess (fit sur x_tr)
      - alloc/ts embeddings (fit sur x_tr)
      - DAE latent (optionnel, fit sur Xtr)
    """
    # num preproc
    num_pre = make_num_pre(params["win_low"], params["win_up"], params["scaler"])
    num_pre_fit = clone(num_pre).fit(x_tr_df[num_cols])
    Xtr_num = num_pre_fit.transform(x_tr_df[num_cols])
    Xte_num = num_pre_fit.transform(x_te_df[num_cols])

    # embeddings
    alloc_enc = GroupSVDEncoder(group_col="ALLOCATION", feature_cols=num_cols,
                                n_components=params["R_ALLOC"], suffix="alloc").fit(x_tr_df)
    x_tr_e = alloc_enc.transform(x_tr_df)
    x_te_e = alloc_enc.transform(x_te_df)
    alloc_cols = [c for c in x_tr_e.columns if c.startswith("alloc_svd_")] + ["alloc_svd_ratio"]

    ts_enc = GroupSVDEncoderSafe(group_col="TS", feature_cols=num_cols,
                             n_components=params["R_TS"], suffix="ts").fit(x_tr_df)
    x_tr_e = ts_enc.transform(x_tr_e)
    x_te_e = ts_enc.transform(x_te_e)
    ts_cols = [c for c in x_tr_e.columns if c.startswith("ts_svd_")] + ["ts_svd_ratio"]

    Xtr = np.column_stack([Xtr_num, x_tr_e[alloc_cols].to_numpy(), x_tr_e[ts_cols].to_numpy()])
    Xte = np.column_stack([Xte_num, x_te_e[alloc_cols].to_numpy(), x_te_e[ts_cols].to_numpy()])

    # DAE (optionnel)
    if params["use_dae"]:
        dae = DAETransformer(
            n_hidden=params["dae_hidden"], code_dim=params["dae_code"],
            noise_std=params["dae_noise"], epochs=params["dae_epochs"],
            batch_size=1024, lr=1e-3, return_code_only=True, random_state=42
        )
        dae_fit = clone(dae).fit(Xtr)
        Ztr = dae_fit.transform(Xtr)
        Zte = dae_fit.transform(Xte)
        Xtr = np.column_stack([Xtr, Ztr])
        Xte = np.column_stack([Xte, Zte])

    return Xtr, Xte

# ---------- objective ----------
def objective(trial: optuna.Trial):
    # espace d'hyperparamètres (budget court, 30 essais)
    params = {
        "R_ALLOC": trial.suggest_categorical("R_ALLOC", [6, 8, 12]),
        "R_TS":    trial.suggest_categorical("R_TS",    [6, 8, 12]),
        "use_dae": trial.suggest_categorical("use_dae", [True, False]),
        "dae_code":  trial.suggest_categorical("dae_code",  [16, 32, 48]) if trial.params.get("use_dae", True) else 16,
        "dae_hidden":trial.suggest_categorical("dae_hidden",[96, 128])    if trial.params.get("use_dae", True) else 96,
        "dae_noise": trial.suggest_float("dae_noise", 0.03, 0.07)         if trial.params.get("use_dae", True) else 0.05,
        "dae_epochs":trial.suggest_categorical("dae_epochs", [8, 12])     if trial.params.get("use_dae", True) else 8,
        "max_keep": trial.suggest_int("max_keep", 160, 420, step=40),
        "stability_q": trial.suggest_categorical("stability_q", [0.3, 0.4, 0.5]),
        "col_thresh": trial.suggest_float("col_thresh", 0.9, 0.98),
        "win_low": trial.suggest_categorical("win_low", [0.002, 0.005]),
        "win_up":  trial.suggest_categorical("win_up",  [0.995, 0.998]),
        "scaler":  trial.suggest_categorical("scaler", ["robust", "standard"]),
    }

    # ridge alpha grid (fixe et large), on pourra raffiner autour du meilleur ensuite
    alpha_grid = np.logspace(-6, 6, 49)

    # CV
    gkf = GroupKFold(n_splits=5)
    rmse_list = []
    for fold_id, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_full, groups=X_train["TS"]), start=1):
        x_tr = X_train.iloc[tr_idx]
        y_tr = y_full[tr_idx]
        x_va = X_train.iloc[va_idx]
        y_va = y_full[va_idx]

        # design
        Xtr, Xva = build_design(x_tr, x_va, params)

        # sélection robuste
        feat_names = [f"f_{i}" for i in range(Xtr.shape[1])]
        Xtr_df = pd.DataFrame(Xtr, columns=feat_names)
        Xva_df = pd.DataFrame(Xva, columns=feat_names)

        selector = RobustFeatureSelector(
            groups=None, k_folds=3,
            col_thresh=params["col_thresh"],
            stability_q=params["stability_q"],
            max_keep=params["max_keep"],
            standardize=True, random_state=42
        )
        Xtr_sel = selector.fit_transform(Xtr_df, y_tr)
        Xva_sel = selector.transform(Xva_df)

        # ridge
        ridge = RidgeCV(alphas=alpha_grid, cv=5, scoring="neg_mean_squared_error", fit_intercept=True)
        ridge.fit(Xtr_sel, y_tr)
        p = ridge.predict(Xva_sel)
        rmse = float(np.sqrt(mean_squared_error(y_va, p)))
        rmse_list.append(rmse)

        # pruning simple après 2 folds
        if fold_id >= 2:
            intermediate = np.mean(rmse_list)
            trial.report(intermediate, step=fold_id)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return float(np.mean(rmse_list))

# ---------- étude ----------
sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=10)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=6, show_progress_bar=True)

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

best_params = study.best_params

# ---------- re-fit final avec la meilleure config et prédire X_test (ensemble 5 folds) ----------
gkf_final = GroupKFold(n_splits=5)
test_preds = []
fold_alphas = []

for tr_idx, _ in gkf_final.split(X_train, y_full, groups=X_train["TS"]):
    x_tr = X_train.iloc[tr_idx]
    y_tr = y_full[tr_idx]

    # design (train + test)
    Xtr, Xte = build_design(x_tr, X_test, best_params)

    # sélection robuste
    feat_names = [f"f_{i}" for i in range(Xtr.shape[1])]
    Xtr_df = pd.DataFrame(Xtr, columns=feat_names)
    Xte_df = pd.DataFrame(Xte, columns=feat_names)

    selector = RobustFeatureSelector(
        groups=None, k_folds=3,
        col_thresh=best_params["col_thresh"],
        stability_q=best_params["stability_q"],
        max_keep=best_params["max_keep"],
        standardize=True, random_state=42
    )
    Xtr_sel = selector.fit_transform(Xtr_df, y_tr)
    Xte_sel = selector.transform(Xte_df)

    ridge = RidgeCV(alphas=np.logspace(-6, 6, 49), cv=5,
                    scoring="neg_mean_squared_error", fit_intercept=True)
    ridge.fit(Xtr_sel, y_tr)
    fold_alphas.append(float(ridge.alpha_))

    test_preds.append(ridge.predict(Xte_sel))

y_test_pred = np.vstack(test_preds).mean(axis=0)
print(f"[FINAL] mean alpha*={np.mean(fold_alphas):.4g} ± {np.std(fold_alphas):.2g}")

# ---------- submission ----------
assert "ROW_ID" in X_test.columns, "La colonne ROW_ID doit exister dans X_test."
sub = pd.DataFrame({
    "ROW_ID": X_test["ROW_ID"].values,
    "prediction": np.nan_to_num(y_test_pred, nan=0.0, posinf=0.0, neginf=0.0).astype(float)
})
sub.to_csv("submission_optuna_best.csv", index=False)
print(sub.head(), "\n[OK] submission_optuna_best.csv écrit.")


[I 2025-10-11 16:01:24,143] A new study created in memory with name: no-name-05752552-aad3-4eec-b078-45a359f48efe


  0%|          | 0/6 [00:00<?, ?it/s]

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/optuna/distributions.py:702: UserWarning: The distribution is specified by [160, 420] and step=40, but the range is not divisible by `step`. It will be replaced by [160, 400].
  warnings.warn(
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning:

[I 2025-10-11 16:04:25,683] Trial 0 finished with value: 0.0016040972358873685 and parameters: {'R_ALLOC': 8, 'R_TS': 6, 'use_dae': False, 'max_keep': 320, 'stability_q': 0.5, 'col_thresh': 0.9665954112640337, 'win_low': 0.002, 'win_up': 0.998, 'scaler': 'robust'}. Best is trial 0 with value: 0.0016040972358873685.


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/alexandrem

: 